# Lok Sabha Winners Scraper — 2014, 2019, and 2024

Scrape the constituency-wise Lok Sabha winners for **Andhra Pradesh, Bihar, and Gujarat** across **2014, 2019, and 2024**.

The scraper uses **manual table selection**: first inspect the tables on each Wikipedia page, then enter the table index and exact column headings for each year/state. The final dataset contains only:

- `State`
- `Election Year`
- `Constituency No.`
- `Constituency`
- `Winner Candidate`

**Special case:** The 2014 Andhra Pradesh page can include Telangana rows at the start. A configurable setting lets you skip the first 18 rows for that page only. Do not apply that skip to 2019 or 2024 unless inspection shows it is needed.

## 2. Imports and page configuration

In [1]:
from io import StringIO
import time
import requests
import pandas as pd
from IPython.display import display

YEARS = [2014, 2019, 2024]
STATES = ["Andhra Pradesh", "Bihar", "Gujarat"]

# Wikipedia article URLs for each year and state.
URLS = {
    (2014, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Andhra_Pradesh",
    (2014, "Bihar"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Bihar",
    (2014, "Gujarat"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Gujarat",
    (2019, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Andhra_Pradesh",
    (2019, "Bihar"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Bihar",
    (2019, "Gujarat"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Gujarat",
    (2024, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Andhra_Pradesh",
    (2024, "Bihar"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Bihar",
    (2024, "Gujarat"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Gujarat",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; EducationalElectionResultsScraper/1.0)"
}

def fetch_page(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text

def flatten_columns(columns):
    """Convert regular or multi-level column headers into readable strings."""
    result = []
    for col in columns:
        parts = col if isinstance(col, tuple) else (col,)
        parts = [
            str(part).strip()
            for part in parts
            if str(part).strip()
            and not str(part).strip().startswith("Unnamed:")
        ]
        result.append(" | ".join(parts) if parts else "Unnamed")
    return result

print(f"Configured {len(URLS)} pages for {len(YEARS)} election years and {len(STATES)} states.")

Configured 9 pages for 3 election years and 3 states.


## 3. Download pages and inspect tables

Run this cell first. For each page it prints table indexes, dimensions, and column headings. Table indexes are zero-based: the first table is `0`.

If a URL returns a 404, check the article link in Wikipedia and update that entry in `URLS`.

In [3]:
PAGE_HTML = {}
PAGE_TABLES = {}
PAGE_ERRORS = {}

for (year, state), url in URLS.items():
    key = (year, state)
    print(f"\n{'=' * 100}\n{year} — {state}\nURL: {url}\n{'=' * 100}")

    try:
        html = fetch_page(url)
        PAGE_HTML[key] = html
        tables = pd.read_html(StringIO(html))
        PAGE_TABLES[key] = tables

        print(f"Found {len(tables)} tables.")
        for index, table in enumerate(tables):
            print(f"\nTable index: {index} | shape: {table.shape}")
            print("Columns:", flatten_columns(table.columns))
            display(table.head(2))
    except Exception as exc:
        PAGE_ERRORS[key] = str(exc)
        # print(f"ERROR loading this page: {exc}")

    time.sleep(1)  # Be polite to Wikipedia

if PAGE_ERRORS:
    print("\nPages needing attention:")
    for key, error in PAGE_ERRORS.items():
        print(f"{key}: {error}")


2014 — Andhra Pradesh
URL: https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Andhra_Pradesh
Found 18 tables.

Table index: 0 | shape: (32, 4)
Columns: ['0', '1', '2', '3']


,0,1,2,3
0,NaN,NaN,NaN,NaN
1,← 2009 30 April 2014 & 7 May 2014 (phases 7 & ...,← 2009 30 April 2014 & 7 May 2014 (phases 7 & ...,NaN,NaN



Table index: 1 | shape: (23, 4)
Columns: ['Unnamed', 'First party', 'Second party', 'Third party']


,Unnamed: 0,First party,Second party,Third party
0,NaN,NaN,NaN,NaN
1,Leader,N. Chandrababu Naidu,K. Chandrashekar Rao,Y. S. Jagan Mohan Reddy



Table index: 2 | shape: (1, 2)
Columns: ['0', '1']


,0,1
0,Prime Minister before election Manmohan Singh INC,Prime Minister after election Narendra Modi BJP


... (304 outputs left out)


Table index: 19 | shape: (6, 2)
Columns: ['vteElections in Gujarat', 'vteElections in Gujarat.1']


,vteElections in Gujarat,vteElections in Gujarat.1
0,General elections,1962 1967 1971 1977 1980 1984 1989 1991 1996 1...
1,Legislative Assembly,1962 1967 1972 1975 1980 1985 1990 1995 1998 2...



Table index: 20 | shape: (2, 2)
Columns: ['0', '1']


,0,1
0,Municipal,... 2010 2015 2021 2026
1,Panchayat,... 2010 2011 2015 2016 2021 2026


## 4. Manual configuration — table indexes, column headings, and row skips

After Step 3, edit `TABLE_INDEX` and `COLUMN_MAP` below.

- `TABLE_INDEX[(year, state)]`: numeric table index for that page.
- `COLUMN_MAP[(year, state)]`: maps exact heading from Step 3 → desired output field.
- `SKIP_ROWS[(year, state)]`: number of leading data rows to skip. The default below skips 18 rows only for **2014 Andhra Pradesh**, matching your earlier setup.

The column heading strings below are examples based on the 2014 table. Update them if headings differ for a year/state.

In [4]:
# 1) Replace None with the correct table index for every page after inspecting Step 3.
TABLE_INDEX = {
    (2014, "Andhra Pradesh"): 9,
    (2014, "Bihar"): 9,
    (2014, "Gujarat"): 7,
    (2019, "Andhra Pradesh"): 6,
    (2019, "Bihar"): 10,
    (2019, "Gujarat"): 7,
    (2024, "Andhra Pradesh"): 13,
    (2024, "Bihar"): 14,
    (2024, "Gujarat"): 14,
    
}

# 2) Exact source heading -> final output column.
# Update each mapping to match the headings printed in Step 3.
DEFAULT_COLUMN_MAP = {
    "Constituency | #": "Constituency No.",
    "Constituency | Name": "Constituency",
    "Winner | Candidate": "Winner Candidate",
}

COLUMN_MAP = {
    (2014, "Andhra Pradesh"): DEFAULT_COLUMN_MAP.copy(),
    (2014, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2014, "Gujarat"): DEFAULT_COLUMN_MAP.copy(),
    (2019, "Andhra Pradesh"): {
        "Constituency | Constituency": "Constituency No.",
        "Constituency | Constituency.1": "Constituency",
        "Winner | Candidate": "Winner Candidate",
    },
    (2019, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2019, "Gujarat"): {
        "Constituency" : "Constituency",
        "No": "Constituency No.",
        "Winner": "Winner Candidate",
    },
    (2024, "Andhra Pradesh"): DEFAULT_COLUMN_MAP.copy(),
    (2024, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2024, "Gujarat"): DEFAULT_COLUMN_MAP.copy(),
}

# 3) Number of leading data rows to skip per page.
# This preserves your earlier Andhra Pradesh 2014 workaround.
SKIP_ROWS = {
    (2014, "Andhra Pradesh"): 17,
    (2014, "Bihar"): 0,
    (2014, "Gujarat"): 0,
    (2019, "Andhra Pradesh"): 0,
    (2019, "Bihar"): 0,
    (2019, "Gujarat"): 0,
}

STRICT_COLUMN_CHECK = True

## 5. Select the manually specified tables and validate headings

This cell does not guess which table is correct. It uses your table indexes and reports missing column names. If it raises an error, correct the relevant `TABLE_INDEX` or `COLUMN_MAP` entry in Step 4.

In [5]:
SELECTED_TABLES = {}

for key, url in URLS.items():
    year, state = key

    if key not in PAGE_TABLES:
        print(f"Skipping {year} {state}: page did not load successfully.")
        continue

    index = TABLE_INDEX.get(key)
    if index is None:
        raise ValueError(
            f"Set TABLE_INDEX[{key!r}] in Step 4 after inspecting the page tables."
        )

    tables = PAGE_TABLES[key]
    if not isinstance(index, int) or not (0 <= index < len(tables)):
        raise IndexError(
            f"{year} {state}: table index {index!r} is invalid. "
            f"Choose an index from 0 to {len(tables) - 1}."
        )

    table = tables[index].copy()
    table.columns = flatten_columns(table.columns)

    skip = SKIP_ROWS.get(key, 0)
    if skip:
        table = table.iloc[skip:].reset_index(drop=True)

    mapping = COLUMN_MAP.get(key, {})
    available = list(table.columns)
    missing = [heading for heading in mapping if heading not in available]

    print(f"\n{year} — {state}: table {index}, shape after row skip={table.shape}")
    print("Selected headings:", available)

    if missing:
        message = f"{year} {state}: headings not found: {missing}"
        if STRICT_COLUMN_CHECK:
            raise ValueError(message + "\nUpdate COLUMN_MAP with exact headings from Step 3.")
        print("WARNING:", message)

    SELECTED_TABLES[key] = table
    display(table.head(3))


2014 — Andhra Pradesh: table 9, shape after row skip=(26, 15)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Poll% | Poll%', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Poll% | Poll%,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,18.0,Araku,72.92,Kothapalli Geetha,NaN,YCP,413191.0,45.42,Gummadi Sandhya Rani,NaN,TDP,259557.0,24.59,171290.0,16.38
1,19.0,Srikakulam,74.36,Ram Mohan Naidu Kinjarapu,NaN,TDP,556163.0,52.90,Reddy Shanthi,NaN,YCP,428591.0,40.76,127572.0,12.14
2,20.0,Vizianagaram,79.79,Ashok Gajapathi Raju,NaN,TDP,536549.0,47.89,Ravu Venkata Swetha,NaN,YCP,429638.0,38.35,106911.0,9.54



2014 — Bihar: table 9, shape after row skip=(41, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Valmiki Nagar,Satish Chandra Dubey,NaN,BJP,364013.0,40.44,Purnmasi Ram,NaN,INC,246218.0,27.35,117795.0,13.09
1,2.0,Paschim Champaran,Sanjay Jaiswal,NaN,BJP,371232.0,43.43,Prakash Jha,NaN,JD(U),260978.0,30.53,110254.0,12.90
2,3.0,Purvi Champaran,Radha Mohan Singh,NaN,BJP,400452.0,48.68,Binod Kumar Srivastava,NaN,RJD,208289.0,25.32,192163.0,23.36



2014 — Gujarat: table 7, shape after row skip=(27, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Kachchh,Vinodbhai Chavda,NaN,BJP,562855.0,59.40,Dinesh Parmar,NaN,INC,308373.0,32.55,254482.0,26.85
1,2.0,Banaskantha,Haribhai Chaudhary,NaN,BJP,507856.0,57.23,Joitabhai Patel,NaN,INC,305522.0,34.43,202334.0,22.80
2,3.0,Patan,Liladhar Vaghela,NaN,BJP,518538.0,54.21,Bhavsinh Rathod,NaN,INC,379819.0,39.70,138719.0,14.51


... (8 outputs left out)


2024 — Bihar: table 14, shape after row skip=(40, 15)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Constituency | Poll\xa0%', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Constituency | Poll %,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1,Valmiki Nagar,60.33,Sunil Kumar,NaN,JD(U),523422,47.50,Deepak Vadav,NaN,RJD,424747,38.55,98675,8.95
1,2,Paschim Champaran,61.78,Sanjay Jaiswal,NaN,BJP,580421,53.43,Madan Mohan Tiwari,NaN,INC,443853,40.86,136568,12.57
2,3,Purvi Champaran,59.90,Radha Mohan Singh,NaN,BJP,580421,50.50,Rajesh Kumar,NaN,VIP,443853,42.28,88287,8.22



2024 — Gujarat: table 14, shape after row skip=(27, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Kachchh,Vinodbhai Chavda,NaN,BJP,659574,60.18,Nitishbhai Lalan,NaN,INC,390792,35.66,268782,24.52
1,2.0,Banaskantha,Geni Thakor,NaN,INC,671883,48.75,Rekhaben Chaudhary,NaN,BJP,641477,46.55,30406,2.20
2,3.0,Patan,Bharatsinhji Dabhi,NaN,BJP,591947,49.51,Chandanji Thakor,NaN,INC,560071,46.85,31876,2.66


## 6. Extract the five required columns and combine all years

Final columns are ordered as `State`, `Election Year`, `Constituency No.`, `Constituency`, and `Winner Candidate`.

In [6]:
def extract_results(key, table):
    year, state = key
    mapping = COLUMN_MAP[key]

    required_outputs = {
        "Constituency No.",
        "Constituency",
        "Winner Candidate",
    }
    missing_outputs = required_outputs - set(mapping.values())
    if missing_outputs:
        raise ValueError(
            f"{year} {state}: COLUMN_MAP must produce these fields: "
            f"{sorted(required_outputs)}. Missing: {sorted(missing_outputs)}"
        )

    missing_inputs = [heading for heading in mapping if heading not in table.columns]
    if missing_inputs:
        raise KeyError(
            f"{year} {state}: missing source headings {missing_inputs}. "
            f"Available: {list(table.columns)}"
        )

    # Select and rename the configured source columns.
    result = table[list(mapping.keys())].rename(columns=mapping)

    # Clean whitespace in text values.
    for col in result.columns:
        result[col] = result[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    result = result.dropna(how="all").copy()
    result.insert(0, "Election Year", year)
    result.insert(0, "State", state)

    return result[
        [
            "State",
            "Election Year",
            "Constituency No.",
            "Constituency",
            "Winner Candidate",
        ]
    ].reset_index(drop=True)


results_by_page = {
    key: extract_results(key, table)
    for key, table in SELECTED_TABLES.items()
}

combined_df = pd.concat(results_by_page.values(), ignore_index=True)

print("Combined dataset shape:", combined_df.shape)
display(combined_df.head(15))
print("\nRows by year and state:")
display(combined_df.groupby(["Election Year", "State"]).size().rename("Rows").to_frame())

Combined dataset shape: (273, 5)


,State,Election Year,Constituency No.,Constituency,Winner Candidate
0,Andhra Pradesh,2014,18.0,Araku,Kothapalli Geetha
1,Andhra Pradesh,2014,19.0,Srikakulam,Ram Mohan Naidu Kinjarapu
2,Andhra Pradesh,2014,20.0,Vizianagaram,Ashok Gajapathi Raju
3,Andhra Pradesh,2014,21.0,Visakhapatnam,Kambhampati Hari Babu
4,Andhra Pradesh,2014,22.0,Anakapalle,Muttamsetti Srinivasa Rao
5,Andhra Pradesh,2014,23.0,Kakinada,Thota Narasimham
6,Andhra Pradesh,2014,24.0,Amalapuram,Pandula Ravindra Babu
7,Andhra Pradesh,2014,25.0,Rajahmundry,Murali Mohan
8,Andhra Pradesh,2014,26.0,Narasapuram,Gokaraju Ganga Raju
9,Andhra Pradesh,2014,27.0,Eluru,Maganti Venkateswara Rao



Rows by year and state:


Rows
Election Year State               
2014          Andhra Pradesh    25
              Bihar             40
              Gujarat           26
2019          Andhra Pradesh    25
              Bihar             40
              Gujarat           26
2024          Andhra Pradesh    25
              Bihar             40
              Gujarat           26

## 7. Export to CSV and Excel

In [ ]:
# CSV_PATH = "loksabha_winners_2014_2019.csv"
# EXCEL_PATH = "loksabha_winners_2014_2019.xlsx"

# combined_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
# combined_df.to_excel(EXCEL_PATH, index=False)

# print("Saved:", CSV_PATH)
# print("Saved:", EXCEL_PATH)

Saved: loksabha_winners_2014_2019.csv
Saved: loksabha_winners_2014_2019.xlsx


## 8. Validation checks

Check that each year/state has the expected number of constituencies and review missing values. Counts can vary if the source table includes by-elections, notes, or other non-constituency rows, so inspect any unexpected result.

In [7]:
display(
    combined_df.groupby(["Election Year", "State"])
    .agg(
        Rows=("Constituency", "size"),
        Missing_Constituency=("Constituency", lambda s: s.isna().sum()),
        Missing_Winner=("Winner Candidate", lambda s: s.isna().sum()),
    )
)

print("\nMissing values per output column:")
display(combined_df.isna().sum().rename("Missing values").to_frame())

print("\nFull preview:")
display(combined_df)

Rows  Missing_Constituency  Missing_Winner
Election Year State                                                     
2014          Andhra Pradesh    25                     0               0
              Bihar             40                     0               0
              Gujarat           26                     0               0
2019          Andhra Pradesh    25                     0               0
              Bihar             40                     0               0
              Gujarat           26                     0               0
2024          Andhra Pradesh    25                     0               0
              Bihar             40                     0               0
              Gujarat           26                     0               0


Missing values per output column:


,Missing values
State,0
Election Year,0
Constituency No.,0
Constituency,0
Winner Candidate,0



Full preview:


,State,Election Year,Constituency No.,Constituency,Winner Candidate
0,Andhra Pradesh,2014,18.0,Araku,Kothapalli Geetha
1,Andhra Pradesh,2014,19.0,Srikakulam,Ram Mohan Naidu Kinjarapu
2,Andhra Pradesh,2014,20.0,Vizianagaram,Ashok Gajapathi Raju
3,Andhra Pradesh,2014,21.0,Visakhapatnam,Kambhampati Hari Babu
4,Andhra Pradesh,2014,22.0,Anakapalle,Muttamsetti Srinivasa Rao
...,...,...,...,...,...
268,Gujarat,2024,22.0,Bharuch,Mansukhbhai Vasava
269,Gujarat,2024,23.0,Bardoli,Parbhubhai Vasava
270,Gujarat,2024,23.0,Surat,Mukesh Dalal
271,Gujarat,2024,25.0,Navsari,C. R. Patil


## Troubleshooting
- **A page URL returns 404:** check Wikipedia for the correct article title and update `URLS`.
- **Wrong table:** use Step 3 to inspect indexes and set the correct `TABLE_INDEX`.
- **Column heading error:** copy the exact heading string from Step 3 into the relevant `COLUMN_MAP` entry.
- **Andhra Pradesh 2014 rows:** this notebook skips the first 18 rows only for that page, as in your earlier workflow. Confirm this is still correct for the table you select.
- **2019/2024 Andhra Pradesh:** no rows are skipped by default.
- **Duplicate rows or unexpected row counts:** inspect the source table and adjust `SKIP_ROWS` only where justified.
- Validate winners against official Election Commission data before using the dataset for analysis.